# Chapter 14: Administration and DevOps

**Level 3: Integration & Engineering** · ⏱ about 4 hours · Dataset: *EduGraph*

### What you will learn

1. Stardog's **security model**: users, roles and permissions
2. **Least privilege** in practice: a read-only user for the API, and what it can and can't do
3. **Named-graph security**: per-graph access, and a trap that can lock everyone out
4. **Operations**: database options, running queries, and **backups** (logical export and restore)
5. **Automated, repeatable deployment** with one script, including a smoke test and teardown
6. **Testing** with pytest: every test in its own temporary named graphs
7. **Mini-project 3, the EduGraph API**: least-privilege Stardog user + API key + tests + deploy script

### What this chapter changes on your server

It creates **users, roles and stored queries** whose names start with `tutorial_`, and briefly switches on **named-graph security** (14.3). The clean-up removes all of them and restores the option.

In [1]:
import importlib
import os
import subprocess
import sys
import tempfile
from dataclasses import replace
from pathlib import Path

TUTORIAL = Path('..').resolve()
for p in (str(TUTORIAL / 'src'), str(TUTORIAL), str(TUTORIAL / 'ops')):
    if p not in sys.path:
        sys.path.insert(0, p)

import pandas as pd
import stardog

from kg import backup, client, dboptions, security, sparql, stored_queries
import deploy_edugraph

pd.set_option('display.max_colwidth', 90)
SETTINGS = client.load_settings()
conn = client.connect()

def attempt(label, fn):
    '''Run fn and report OK, or the HTTP error code, as a table row.'''
    try:
        result = fn()
        return {'action': label, 'outcome': 'OK', 'detail': str(result)[:70]}
    except stardog.exceptions.StardogException as e:
        return {'action': label, 'outcome': f'HTTP {e.http_code}', 'detail': str(e).split(': ', 1)[-1][:70]}

## 14.1 The security model

Stardog authorizes every request against **permissions**. A permission is a triple of its own:

> **action** × **resource type** × **resource**, e.g. *read* × *db* × *mysampledb*

| Actions | Resource types (the common ones) |
|---|---|
| `read`, `write`, `create`, `delete`, `execute`, `grant`, `revoke`, `all` | `db` (a database), `named-graph` (`db\graph`), `stored-query` (by name), `metadata` (database options), `user`, `role`, `admin` (server tasks), `*` (everything) |

**Roles** bundle permissions, and **users** get roles plus their own permissions. A **superuser** bypasses all checks. Your user's permissions (Chapter 2):

In [2]:
with client.admin() as admin:
    me = admin.user(SETTINGS.username)
    print(f'{me.name}: roles {[r.name for r in me.roles()]}, superuser: {me.is_superuser()}')
    perms = pd.DataFrame(me.effective_permissions())
perms.assign(resource=perms.resource.map(lambda r: '\\'.join(r))).groupby('resource_type').action.apply(
    lambda a: ', '.join(sorted(set(a)))).to_frame('actions')

rakesh: roles ['cloud'], superuser: False


,actions
resource_type,
*,"CREATE, DELETE, EXECUTE, GRANT, READ, REVOKE, WRITE"
admin,"CREATE, DELETE, EXECUTE, GRANT, READ, REVOKE, WRITE"
cache,"CREATE, DELETE, EXECUTE, GRANT, READ, REVOKE, WRITE"
cache-target,"CREATE, DELETE, EXECUTE, GRANT, READ, REVOKE, WRITE"
db,"CREATE, DELETE, EXECUTE, GRANT, READ, REVOKE, WRITE"
db-export,"CREATE, DELETE, EXECUTE, GRANT, READ, REVOKE, WRITE"
entity-resolution,"CREATE, DELETE, EXECUTE, GRANT, READ, REVOKE, WRITE"
icv-constraints,"CREATE, DELETE, EXECUTE, GRANT, READ, REVOKE, WRITE"
metadata,"CREATE, DELETE, EXECUTE, GRANT, READ, REVOKE, WRITE"


The `cloud` role gives broad rights on everything (`*`). Perfect for an administrator, far too much for an application.

## 14.2 Least privilege: a read-only user for the API

An application should connect as **its own user** with only what it needs. If the API is compromised, the attacker can then only *read* what the API could read anyway.

`kg.security` manages roles and users idempotently (running it twice gives the same result):

In [3]:
print(Path(security.__file__).read_text())

r"""Users, roles and permissions, managed idempotently (Chapter 14).

A permission is (action, resource type, resource), e.g. ('read', 'db', 'mysampledb'),
('read', 'stored-query', 'my_query') or ('read', 'named-graph', 'mysampledb\urn:my:graph').
"""
import secrets
import string

import stardog

from kg import client


def new_password(length: int = 24) -> str:
  """A random password of letters and digits (Stardog rejects some punctuation)."""
  return ''.join(secrets.choice(string.ascii_letters + string.digits) for _ in range(length))


def _exists(fetch) -> bool:
  try:
    fetch()
    return True
  except stardog.exceptions.StardogException as e:
    if e.http_code == 404:
      return False
    raise


def ensure_role(name: str, permissions: list[tuple[str, str, str]]) -> None:
  """Create the role, or re-create it, with exactly these permissions."""
  with client.admin() as admin:
    if _exists(lambda: admin.role(name).permissions()):
      admin.role(name).delete(force=True)   

A role that may only **read the database**, and a user with that role. Passwords are generated (letters and digits; Stardog rejects some punctuation), and never written into notebooks or code:

In [4]:
DEMO_ROLE, DEMO_USER = 'tutorial_ch14_reader', 'tutorial_ch14_reader_user'
DEMO_PASSWORD = security.new_password()
security.ensure_role(DEMO_ROLE, [('read', 'db', SETTINGS.database)])
security.ensure_user(DEMO_USER, DEMO_PASSWORD, [DEMO_ROLE])
reader = replace(SETTINGS, username=DEMO_USER, password=DEMO_PASSWORD)

DEMO_GRAPH = 'urn:tutorial:ch14:demo'
with client.transaction(conn):
    conn.clear(graph_uri=DEMO_GRAPH)
    conn.add(stardog.content.Raw(b'<urn:tutorial:a> <urn:tutorial:b> "hello" .', 'text/turtle'), graph_uri=DEMO_GRAPH)

with client.connect(settings=reader) as rc, client.admin(reader) as radmin:
    results = [
        attempt('SELECT a query', lambda: sparql.run_query(rc, 'SELECT ?o { ?s ?p ?o }', graphs=DEMO_GRAPH).o.tolist()),
        attempt('INSERT data', lambda: rc.update(f'INSERT DATA {{ GRAPH <{DEMO_GRAPH}> {{ <urn:x> <urn:y> "z" }} }}')),
        attempt('clear a graph', lambda: (rc.begin(), rc.clear(graph_uri=DEMO_GRAPH), rc.commit())),
        attempt('list databases', lambda: [d.name for d in radmin.databases()]),
        attempt('list users', lambda: [u.name for u in radmin.users()]),
        attempt('create a database', lambda: radmin.new_database('tutorial_should_fail')),
    ]
    rc.transaction = None          # the failed clear left pystardog thinking a transaction is open
pd.DataFrame(results)

,action,outcome,detail
0,SELECT a query,OK,['hello']
1,INSERT data,HTTP 403,com.complexible.stardog.security.StardogAuthorizationException: User d
2,clear a graph,HTTP 403,User does not have write permissions for the index. User identificati
3,list databases,OK,['mysampledb']
4,list users,OK,['tutorial_ch14_reader_user']
5,create a database,HTTP 403,"Permission [[create], [db], [*]] denied for tutorial_ch14_reader_user"


- **Reads work; writes are refused** (403).
- Listing is **filtered**: the user sees only the databases and users it may see (itself).

### Stored queries need their own permission

The API runs **stored queries** (Chapter 13). Reading the database isn't enough:

In [5]:
QN = 'tutorial_ch14_count'
with client.admin() as admin:
    if QN in stored_queries.list_names():
        admin.stored_query(QN).delete()
    admin.new_stored_query(QN, 'SELECT (COUNT(*) AS ?n) { ?s ?p ?o }', {'database': SETTINGS.database})

def run_as_reader():
    with client.connect(settings=reader) as rc:
        return sparql.run_stored(rc, QN, graphs=DEMO_GRAPH).n[0]

before = attempt('run stored query (db read only)', run_as_reader)
security.ensure_role(DEMO_ROLE, [('read', 'db', SETTINGS.database), ('read', 'stored-query', QN)])
security.ensure_user(DEMO_USER, DEMO_PASSWORD, [DEMO_ROLE])     # re-creating a role detaches it: re-attach
after = attempt('run stored query (+ read on that query)', run_as_reader)
pd.DataFrame([before, after])

,action,outcome,detail
0,run stored query (db read only),HTTP 403,"Permission [[read], [stored-query], [*]] denied for tutorial_ch14_read"
1,run stored query (+ read on that query),OK,1


Permission is granted **per query name**, so the API user can run exactly the queries it serves and no others.

Note the second `ensure_user` call: `ensure_role` re-creates the role, which **detaches it from its users**, so the user is re-attached right after. `deploy_edugraph.py` does the same.

> 🔁 **Neo4j equivalent:** Neo4j Enterprise has roles with `GRANT MATCH`, `GRANT WRITE`, per-label and per-property privileges. The least-privilege principle is identical.

## 14.3 Named-graph security

So far permissions are **per database**. With the database option `security.named.graphs` switched on, Stardog also checks **per named graph**. A user then sees only the graphs they have `read` permission on, written as resource type `named-graph` and resource `database\graph`.

That's how one database can hold data for several teams or customers safely (Chapter 12's "search index sees everything" problem is solved on the server).

A demonstration with two graphs and the reader from 14.2, allowed to read **graph A only**. The option is switched on and **always** switched back off (`try/finally`):

In [6]:
A, B = 'urn:tutorial:ch14:ng:a', 'urn:tutorial:ch14:ng:b'
with client.transaction(conn):
    for g, label in ((A, 'secret of team A'), (B, 'secret of team B')):
        conn.clear(graph_uri=g)
        conn.add(stardog.content.Raw(f'<urn:tutorial:{g[-1]}> <urn:tutorial:says> "{label}" .'.encode(), 'text/turtle'), graph_uri=g)

security.ensure_role(DEMO_ROLE, [('read', 'db', SETTINGS.database), ('read', 'named-graph', f'{SETTINGS.database}\\{A}')])
security.ensure_user(DEMO_USER, DEMO_PASSWORD, [DEMO_ROLE])

READ_BOTH = 'SELECT ?said { ?s <urn:tutorial:says> ?said } ORDER BY ?said'
def what_can_they_read(settings):
    with client.connect(settings=settings) as c:
        return sparql.run_query(c, READ_BOTH, graphs=[A, B]).said.tolist()

rows = [{'named-graph security': 'off', 'reader (graph A only)': what_can_they_read(reader), 'you (admin role)': what_can_they_read(SETTINGS)}]
original = dboptions.set_options({'security.named.graphs': True})
try:
    rows.append({'named-graph security': 'on', 'reader (graph A only)': what_can_they_read(reader), 'you (admin role)': what_can_they_read(SETTINGS)})
finally:
    dboptions.set_options(original)
print('restored:', dboptions.get_options('security.named.graphs'))
pd.DataFrame(rows)

restored: {'security.named.graphs': False}


,named-graph security,reader (graph A only),you (admin role)
0,off,"[secret of team A, secret of team B]","[secret of team A, secret of team B]"
1,on,[secret of team A],[]


- With the option **on**, the reader sees **only graph A**. The permission works exactly as intended.
- ⚠️ **Your own user saw nothing either**, despite its broad role. Once named-graph security is on, access to named graphs must be **granted explicitly** to every user who needs it. On a live system, switching it on without preparing permissions **locks users out of their data**.

**Plan before enabling:** list who needs which graphs, grant `read`/`write` on `named-graph` resources (or wildcards) to every role, test with each role, and only then flip the option during a maintenance window.

## 14.4 Operations

### Database options

Options control features (Chapters 2, 8, 12). Some change online, others need the database **offline** (`kg.dboptions.set_options` handles both). A few worth knowing:

In [7]:
pd.DataFrame(dboptions.get_options('reasoning.type', 'search.enabled', 'spatial.enabled', 'security.named.graphs',
                                   'query.all.graphs', 'query.timeout', 'transaction.write.conflict.strategy',
                                   'edge.properties').items(), columns=['option', 'value'])

,option,value
0,query.all.graphs,False
1,transaction.write.conflict.strategy,LAST_COMMIT_WINS
2,reasoning.type,SL
3,spatial.enabled,False
4,security.named.graphs,False
5,search.enabled,False
6,query.timeout,5m
7,edge.properties,False


`query.timeout` is a safety net: no query can run longer than that. Lower it for databases that serve interactive applications.

### What's running right now?

`admin.queries()` lists running queries (with who started them and how long they've run), and `admin.kill_query(id)` stops one. That's the first thing to check when a database suddenly feels slow:

In [8]:
with client.admin() as admin:
    print('healthy:', admin.healthcheck())
    running = admin.queries()
    print('running queries:', running if running else 'none (besides this check)')

healthy: True


running queries: none (besides this check)


### Backups

Two kinds:

| | **Server backup** | **Logical backup** (export) |
|---|---|---|
| How | `admin.database(db).backup()`, `admin.restore(…)` | Export graphs to RDF files, reload them |
| Covers | The whole database: data, options, indexes | The graphs you choose |
| Restore | Replaces the whole database, as an admin operation | Graph by graph, into any Stardog (or any RDF store) |
| On Stardog Cloud | Managed by the service | Your responsibility, under your control |

`kg.backup` does logical backups: each graph to a compressed Turtle file, plus a `manifest.json`:

In [9]:
print(Path(backup.__file__).read_text())

"""Logical backups: export named graphs to compressed Turtle files, and restore them (Chapter 14).

A logical backup is portable (plain RDF files you can load into any Stardog, or any RDF store),
selective (only the graphs you choose) and needs no server file system access.
"""
import gzip
import json
from datetime import datetime, timezone
from pathlib import Path

import stardog

from kg import client


def export_graphs(graphs: list[str], folder: Path) -> Path:
  """Write each graph to <folder>/<timestamp>/NN.ttl.gz plus a manifest.json. Returns the backup folder."""
  target = Path(folder) / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
  target.mkdir(parents=True)
  manifest = []
  with client.connect() as conn:
    for i, graph in enumerate(graphs):
      data = conn.export(content_type='text/turtle', graph_uri=graph)
      name = f'{i:02d}.ttl.gz'
      with gzip.open(target / name, 'wb') as f:
        f.write(data)
      count = conn.select(f'SELECT (COUNT(*) AS ?n) {{ 

Back up the two demo graphs, delete one ("an accident"), and restore:

In [10]:
BACKUPS = Path(tempfile.mkdtemp(prefix='stardog-backup-'))
saved = backup.export_graphs([A, B], BACKUPS)
print('backup written to', saved, '->', sorted(p.name for p in saved.iterdir()))

conn.update(f'DROP GRAPH <{A}>')
print('after the accident:', sparql.run_query(conn, READ_BOTH, graphs=[A, B]).said.tolist())

print('restored:', [(m['graph'], m['triples']) for m in backup.restore(saved)])
print('after restore:     ', sparql.run_query(conn, READ_BOTH, graphs=[A, B]).said.tolist())

backup written to C:\Users\rakes\AppData\Local\Temp\stardog-backup-nyo7sazh\20260927T155550Z -> ['00.ttl.gz', '01.ttl.gz', 'manifest.json']


after the accident: ['secret of team B']


restored: [('urn:tutorial:ch14:ng:a', 1), ('urn:tutorial:ch14:ng:b', 1)]


after restore:      ['secret of team A', 'secret of team B']


Schedule logical backups for anything you can't recreate from source files, and **test restores**. A backup that has never been restored is a hope, not a backup.

## 14.5 Automated deployment

Everything the EduGraph API needs (graphs, stored queries, a role, a user) should be set up by **one script**, not by hand. Then every environment (dev, test, prod) is identical, and a broken setup is fixed by re-running it.

`ops/deploy_edugraph.py`:

In [11]:
print((TUTORIAL / 'ops' / 'deploy_edugraph.py').read_text())

r"""Deploy (or tear down) the EduGraph stack on Stardog, repeatably (Chapter 14).

What 'deploy' does, in order:
  1. check the connection (the Chapter 2 setup checks)
  2. load the curriculum and import the SIS CSVs into this environment's named graphs
  3. publish the EduGraph .rq files as stored queries
  4. create a least-privilege role and API user: read the database, its graphs and its stored queries only
  5. smoke-test: run a stored query as the API user

Running it again replaces everything with the same result (idempotent).

Usage, from the stardog_tutorial folder:
    set EDUGRAPH_STARDOG_PASSWORD=<a password of letters and digits>
    ..\.venv\Scripts\python ops\deploy_edugraph.py deploy   --env demo
    ..\.venv\Scripts\python ops\deploy_edugraph.py teardown --env demo
"""
import argparse
import json
import os
import sys
from dataclasses import replace
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / 'src'))

import stardog

Key ideas:

- **Names derive from the environment** (`--env dev` → graphs `urn:edugraph:dev:…`, user `dev_api`), so environments never collide.
- **Idempotent**: every step replaces rather than adds, so running twice gives the same result.
- **Least privilege** is built in: the role gets `read` on the database, on each of its graphs (ready for named-graph security) and on each stored query, and nothing else.
- A **smoke test** proves the result as the API user: it can read, and it **can't** write.
- **Secrets** (the API user's password) come from the environment, never from the code.

Deploy an environment called `tutorial-ch14`:

In [12]:
ENV = 'tutorial-ch14'
API_PASSWORD = security.new_password()
cfg = deploy_edugraph.deploy(ENV, API_PASSWORD)

1. connection OK


2. graphs loaded: {'curriculum': None, 'students': 280, 'enrollments': 385, 'assessments': 1700}


3. 6 stored queries published


4. role tutorial_ch14_api_reader (11 permissions) and user tutorial_ch14_api ready


5. smoke test OK: the API user reads 21 topics and cannot write


Run it **again**: same outcome, no duplicates, no errors. That's what makes it safe to automate:

In [13]:
cfg = deploy_edugraph.deploy(ENV, API_PASSWORD, log=lambda m: print('  ', m))

   1. connection OK


   2. graphs loaded: {'curriculum': None, 'students': 280, 'enrollments': 385, 'assessments': 1700}


   3. 6 stored queries published


   4. role tutorial_ch14_api_reader (11 permissions) and user tutorial_ch14_api ready


   5. smoke test OK: the API user reads 21 topics and cannot write


From a terminal (or a CI pipeline) it's a single command:

```
set EDUGRAPH_STARDOG_PASSWORD=<generated password>
..\.venv\Scripts\python ops\deploy_edugraph.py deploy   --env prod
..\.venv\Scripts\python ops\deploy_edugraph.py teardown --env prod
```

## 14.6 Testing with pytest

Tests against a knowledge graph need **isolation**: a test must not see another test's data, nor break real data. The fixtures in `tests/conftest.py` give each test its **own temporary named graphs** with random names, and remove them afterwards. That's the named-graph equivalent of "a temporary database per test", and it works on any Stardog plan.

In [14]:
print((TUTORIAL / 'tests' / 'conftest.py').read_text())

"""Shared pytest fixtures (Chapter 14). Every test works in its own temporary named graphs.

Run from the stardog_tutorial folder:   ..\.venv\Scripts\python -m pytest tests -q
"""
import sys
import uuid
from pathlib import Path

import pytest

ROOT = Path(__file__).resolve().parents[1]
sys.path[:0] = [str(ROOT / 'src'), str(ROOT)]

import stardog                            # noqa: E402

from kg import client, imports            # noqa: E402


def unique(name: str) -> str:
  return f'urn:tutorial:test:{uuid.uuid4().hex[:8]}:{name}'


@pytest.fixture(scope='session')
def conn():
  with client.connect() as c:
    yield c


@pytest.fixture
def temp_graph(conn):
  """An empty named graph for one test, removed afterwards."""
  graph = unique('scratch')
  yield graph
  with client.transaction(conn):
    conn.clear(graph_uri=graph)


@pytest.fixture(scope='session')
def edugraph(conn):
  """The EduGraph curriculum and SIS data, loaded once into temporary graphs for the whole session."""
  grap

In [15]:
print((TUTORIAL / 'tests' / 'test_queries.py').read_text())

"""The EduGraph queries give the answers we expect, including on data made up for the test."""
import stardog
from rdflib import URIRef

from kg import client, sparql

TOPIC = 'http://example.org/edu/topic/'
STUDENT = 'http://example.org/edu/student/'


def test_regression_needs_statistics_and_its_foundations(conn, edugraph):
  rows = sparql.run_query(conn, 'edugraph/topic_prerequisites', graphs=list(edugraph.values()),
                          topic=URIRef(TOPIC + 'regression'))
  direct = set(rows[rows.direct].id)
  assert direct == {'statistics', 'linear-algebra', 'pandas'}
  assert {'probability', 'arithmetic', 'python-basics'} <= set(rows.id)      # indirect, several steps back


def _student(conn, graph, score):
  """A made-up student in ML201 with one Statistics score."""
  with client.transaction(conn):
    conn.clear(graph_uri=graph)
    conn.add(stardog.content.Raw(f'''
      @prefix edu: <http://example.org/edu#> .
      <{STUDENT}S900> a edu:Student ; edu:name "Test Studen

The suite tests three layers:

| File | Tests |
|---|---|
| `test_queries.py` | Query results on real data, and on made-up data (a student with a known weak score) |
| `test_validation.py` | The SHACL shapes accept a good recipe and reject a bad one |
| `test_api.py` | The API: key required (401), validation (422), not found (404), correct answers. Its stored queries are published under a random prefix and removed afterwards |

Run it, exactly as CI would:

In [16]:
result = subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q', '-p', 'no:cacheprovider', '--color=no'],
                        cwd=TUTORIAL, capture_output=True, text=True, env={**os.environ, 'PYTHONWARNINGS': 'ignore'})
print(result.stdout[-1500:])
print('exit code:', result.returncode)

............                                                             [100%]
12 passed in 58.62s

exit code: 0


## Mini-project 3: the EduGraph API, production-style

Level 3 ends by assembling the pieces into something you could hand to an operations team:

| Piece | Where |
|---|---|
| REST API over stored queries | `api/main.py` (Chapter 13) |
| **API key** for clients (`X-API-Key` header) | `api/main.py`, setting `EDUGRAPH_API_KEY` |
| **Least-privilege Stardog user** for the API | `deploy_edugraph.py` → `EDUGRAPH_STARDOG_USER` / `_PASSWORD` |
| Repeatable **deploy / teardown** with smoke test | `ops/deploy_edugraph.py` |
| **Tests** with isolated graphs | `tests/` |
| **Backups** | `kg.backup` |

Point the API at the deployment from 14.5, running **as the least-privilege user**, and protected by a key:

In [17]:
from fastapi.testclient import TestClient

API_KEY = security.new_password(32)
os.environ.update(deploy_edugraph.api_environment(cfg, API_PASSWORD, API_KEY))
import api.main
api_app = importlib.reload(api.main).app           # re-read the environment
print('API connects to Stardog as:', api.main.SETTINGS.username)

client_app = TestClient(api_app)
checks = [
    ('GET /health (no key needed)', client_app.get('/health')),
    ('GET /students/S023 without key', client_app.get('/students/S023')),
    ('GET /students/S023 with key', client_app.get('/students/S023', headers={'X-API-Key': API_KEY})),
    ('GET /students/S023/gaps with key', client_app.get('/students/S023/gaps', headers={'X-API-Key': API_KEY})),
]
pd.DataFrame([(label, r.status_code, str(r.json())[:90]) for label, r in checks], columns=['request', 'status', 'body'])

API connects to Stardog as: tutorial_ch14_api


,request,status,body
0,GET /health (no key needed),200,"{'status': 'ok', 'database': 'mysampledb', 'triples': 4478}"
1,GET /students/S023 without key,401,{'detail': 'Missing or wrong X-API-Key header'}
2,GET /students/S023 with key,200,"{'id': 'S023', 'name': 'Aarav Sharma', 'city': 'Sao Paulo', 'year': 2, 'courses': 'MA1..."
3,GET /students/S023/gaps with key,200,"{'id': 'S023', 'threshold': 50, 'gaps': [{'student_name': 'Aarav Sharma', 'weak_topic'..."


The API now runs with **two layers** of protection: clients need the key, and even a compromised API could only **read** its own graphs and run its own queries.

### In CI/CD

A pipeline (e.g. GitHub Actions) would run on every change:

```yaml
jobs:
  test-and-deploy:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: '3.13' }
      - run: pip install pystardog python-dotenv rdflib pandas fastapi httpx pytest
      - run: python -m pytest stardog_tutorial/tests -q          # against a test database
        env: { STARDOG_ENDPOINT: ..., STARDOG_USERNAME: ..., STARDOG_PASSWORD: ${{ secrets.STARDOG_PASSWORD }}, STARDOG_DATABASE: ... }
      - run: python stardog_tutorial/ops/deploy_edugraph.py deploy --env prod
        if: github.ref == 'refs/heads/main'
        env: { EDUGRAPH_STARDOG_PASSWORD: ${{ secrets.EDUGRAPH_API_PASSWORD }}, ... }
```

Credentials live in the CI system's **secret store**, never in the repository.

🎉 **Level 3 complete.** You can integrate structured and unstructured sources, search and locate, serve the graph through APIs, and run it safely.

## 14.7 Exercises

**Exercise 1.** Create a role `tutorial_ch14_writer` that may **write** only to the demo graph `DEMO_GRAPH` (plus read the database), give it to a new user, and check that it can insert into that graph. Remove user and role afterwards.

<details><summary>Solution</summary>

```python
pw = security.new_password()
security.ensure_role('tutorial_ch14_writer', [('read', 'db', SETTINGS.database), ('write', 'db', SETTINGS.database)])
security.ensure_user('tutorial_ch14_writer_user', pw, ['tutorial_ch14_writer'])
with client.connect(settings=replace(SETTINGS, username='tutorial_ch14_writer_user', password=pw)) as wc:
    wc.update(f'INSERT DATA {{ GRAPH <{DEMO_GRAPH}> {{ <urn:w> <urn:w> "written" }} }}')
security.drop(user='tutorial_ch14_writer_user', role='tutorial_ch14_writer')
```
Without named-graph security, `write` is per **database**. Restricting writes to one graph needs `security.named.graphs` on and a `write` permission on the `named-graph` resource.
</details>

**Exercise 2.** Add a test to `tests/test_queries.py`: `course_performance` must return exactly the four courses DB150, DS101, MA120, ML201. Run pytest.

<details><summary>Solution</summary>

```python
def test_four_courses(conn, edugraph):
    rows = sparql.run_query(conn, 'edugraph/course_performance', graphs=list(edugraph.values()))
    assert sorted(rows.course) == ['DB150', 'DS101', 'MA120', 'ML201']
```
</details>

**Exercise 3.** Deploy a second environment `tutorial-ch14-b` next to the first. Check that the two API users can't run each other's stored queries. Tear it down afterwards.

<details><summary>Solution</summary>

```python
pw_b = security.new_password()
cfg_b = deploy_edugraph.deploy('tutorial-ch14-b', pw_b)
with client.connect(settings=replace(SETTINGS, username=cfg_b['user'], password=pw_b)) as c:
    print(attempt('b runs a query of env a', lambda: sparql.run_stored(c, cfg['query_prefix'] + 'topics')))   # HTTP 403
deploy_edugraph.teardown('tutorial-ch14-b')
```
</details>

**Exercise 4.** Your backup job has run nightly for a year and never failed. Why is that not enough?

<details><summary>Solution</summary>

"Didn't fail" only means files were written. Unless you **restore** them regularly (into a scratch database or graphs) and check the result, you don't know that they're complete, readable, or restorable in the time you'd have during an incident.
</details>

In [18]:
# your answers here

## 14.8 Clean up

Tears down the deployment, and removes the demo user, role, stored query, graphs and backup files. Named-graph security was already restored in 14.3.

In [19]:
import shutil

CLEAN_UP = True

if CLEAN_UP:
    deploy_edugraph.teardown(ENV)
    for k in ('EDUGRAPH_GRAPHS', 'EDUGRAPH_QUERY_PREFIX', 'EDUGRAPH_STARDOG_USER', 'EDUGRAPH_STARDOG_PASSWORD', 'EDUGRAPH_API_KEY'):
        os.environ.pop(k, None)
    security.drop(user=DEMO_USER, role=DEMO_ROLE)
    with client.admin() as admin:
        if QN in stored_queries.list_names():
            admin.stored_query(QN).delete()
    with client.transaction(conn):
        for g in (DEMO_GRAPH, A, B):
            conn.clear(graph_uri=g)
    shutil.rmtree(BACKUPS, ignore_errors=True)
    with client.admin() as admin:
        print('users:', [u.name for u in admin.users()])
        print('roles:', [r.name for r in admin.roles()])
    print('stored queries:', stored_queries.list_names())
    print('named-graph security:', dboptions.get_options('security.named.graphs'))
conn.close()

removed 6 stored queries, user tutorial_ch14_api, role tutorial_ch14_api_reader and 4 graphs


users: ['rakesh.panigrahy@chaatashaali.com', 'rakesh', 'admin']


roles: ['cloud', 'reader', 'executor']


stored queries: ['start']


named-graph security: {'security.named.graphs': False}


## Summary

- **Permissions** are *action × resource type × resource*, bundled into **roles** and given to **users**.
- Give applications their **own least-privilege user**: read the database, read **each** stored query it runs, nothing else. Verify with a failing write.
- **Named-graph security** restricts access per graph, but once it's on, **every** user needs explicit graph permissions, including admins with broad roles. Plan permissions before switching it on.
- Operate with **options** (timeouts!), a look at **running queries**, and **backups**. Logical exports are portable and selective. **Test your restores.**
- **Deploy with one idempotent script** that derives names from the environment, applies least privilege, and smoke-tests the result. Keep secrets in the environment.
- **Test** queries, shapes and APIs with **pytest**, each test isolated in temporary named graphs, and run it all in CI.

**Next: Level 4, Chapter 15, Query performance.** Reading query plans, cardinality estimates, join order, and rewriting slow queries, measured on a larger generated dataset.